In [ ]:
# Create baseline models: feature engineering (domain knowledge feature) ---> split dataset ---> encoding ---> modeling ---> evaluate models 
# Imports
import os
import sys
import numpy as np
import pandas as pd
from huggingface_hub import login
# Add the utils folder to Python path
notebook_dir = os.path.dirname(os.path.abspath("__file__"))
pricer_path = os.path.join(notebook_dir, "../utils")
sys.path.insert(0, pricer_path)
# Class Item to validate the returned data and has lots of helper functions
from pricer.items import Item
from pricer.evaluator import evaluate
# Machine learning models
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb 

In [2]:
# Login to Hugging Face
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
# Download the batches dataset from Hugging Face 
username = "omarshaarawy11"
dataset = f"{username}/best_deal_03_batches"
train, val, test = Item.from_hub(dataset)
items = train + val + test
print(f"Loaded {len(items):,} items")

Loaded 21,091 items


In [4]:
# Invistigate the dataset
columns = ['title', 
'category', 
'price',
'full',
'weight',
'summary',
'prompt',
'id'
]
for i in columns:
    # Combine item with its attrs
    attr = getattr(items[0], i)
    print(attr)

Lurrose 100Pcs Full Cover Fake Toenails Artificial Transparent Nail Tips Nail Art for DIY
all beauty
6.99
None
0.0
Title: Lurrose 100pcs Full Cover Fake Toenails  
Category: Nail Art & Accessories  
Brand: Lurrose  
Description: High-quality, durable artificial toenails perfect for DIY nail art and manicures.  
Details: Features 100 lightweight, eco-friendly ABS nails that are easy to trim, shape, and fit comfortably to natural toenails, suitable for various styles and perfect as a gift.
None
None


In [5]:
# Input features for the ML models
features = list(Item.model_fields.keys())
features.remove('price')
print(features)

['title', 'category', 'full', 'weight', 'summary', 'prompt', 'completion', 'id']


In [6]:
# Feature engineering
# Domain knowledge feature
# We need weight, lenght of the whole summary of the product and the column of weight_unknown in case we don't known about weight
def get_features(item):
    # As long as we work with pydantics we can access any attribute of the object using .
    return {
        "weight": item.weight,
        "weight_unknown": 1 if item.weight==0 else 0,
        "text_length": len(item.summary)
    } 

In [7]:
# Split dataset
train = train
test = test

In [8]:
# Convert to dataframe
def list_to_dataframe(items):
    features = [get_features(item) for item in items]
    # Input features
    df = pd.DataFrame(features)
    # Output feature
    df['price'] = [item.price for item in items]
    return df

train_df = list_to_dataframe(train)
test_df = list_to_dataframe(test)

In [9]:
train_df

,weight,weight_unknown,text_length,price
0,0.00000,1,377,6.99
1,0.00000,1,359,5.99
2,0.00000,1,416,29.80
3,0.00000,1,337,24.00
4,0.00000,1,379,22.49
...,...,...,...,...
16795,0.35000,0,349,17.49
16796,1.01000,0,387,25.97
16797,0.36875,0,421,25.99
16798,3.61000,0,366,49.87


In [10]:
test_df

,weight,weight_unknown,text_length,price
0,0.65625,0,353,8.99
1,13.80000,0,347,49.99
2,0.56875,0,353,9.99
3,1.50000,0,319,21.99
4,2.79000,0,365,32.99
...,...,...,...,...
2186,0.00000,1,317,50.00
2187,0.00000,1,330,50.00
2188,0.00000,1,316,12.99
2189,0.03300,0,311,25.00


In [11]:
# Modeling
# Traditional linear regression
# The results of train and test is not good enough so our feature selection is not good enough
np.random.seed(40)
feature_columns = ['weight', 'weight_unknown', 'text_length']
X_train = train_df[feature_columns]
y_train = train_df['price']
X_test = test_df[feature_columns]
y_test = test_df['price']

# Train 
model = LinearRegression()
model.fit(X_train, y_train)
for feature, coef in zip(feature_columns, model.coef_):
    print(f"{feature}: {coef: 0.2f}")
print(f"Intercept: {model.intercept_: 0.2f}")

# Predict 
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# Evaluate 
train_mse = mean_squared_error(y_train, y_train_pred)
train_r2 = r2_score(y_train, y_train_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)

print(f"Train MSE: {train_mse: 0.2f}")
print(f"Train R2 Score: {train_r2: 0.2f}%")
print(f"Test MSE: {test_mse: 0.2f}")
print(f"Test R2 Score: {test_r2: 0.2f}%")

weight:  4.25
weight_unknown:  0.82
text_length:  0.09
Intercept:  12.38
Train MSE:  8252.34
Train R2 Score:  0.24%
Test MSE:  3520.26
Test R2 Score:  0.43%


In [12]:
# Test dataset
def linear_regression_pricer(item):
    features = get_features(item)
    features_df = pd.DataFrame([features])
    # The new coming data points we work with .predict
    # [0] the prediction of the model
    return model.predict(features_df)[0]

In [13]:
# Test single datapoint
linear_regression_pricer(items[0])

45.81016596666426

In [14]:
# Evaluation
evaluate(linear_regression_pricer, test)

  0%|          | 0/2000 [00:00<?, ?it/s]

$37 $51 $35 $24 $23 $26 $27 $22 $30 $33 $16 $23 $34 $45 $36 $38 $31 $19 $130 $4 $34 $31 $17 $29 $38 $30 $31 $50 $29 $15 $25 $34 $28 $561 $32 $27 $23 $27 $88 $26 $31 $41 $35 $12 $14 $34 $32 $30 $37 $14 $34 $36 $33 $30 $29 $31 $7 $11 $35 $35 $50 $31 $45 $33 $36 $26 $41 $17 $34 $22 $32 $28 $30 $25 $13 $19 $18 $9 $7 $167 $3 $26 $42 $31 $21 $38 $31 $38 $24 $33 $32 $31 $39 $39 $12 $35 $28 $15 $38 $25 $32 $22 $36 $31 $40 $36 $34 $7 $4 $21 $19 $13 $17 $100 $25 $43 $25 $46 $34 $7 $10 $9 $30 $24 $13 $0 $5 $31 $33 $27 $31 $34 $99 $212 $28 $29 $25 $35 $43 $32 $31 $16 $35 $31 $35 $34 $31 $4 $27 $34 $2 $12 $27 $13 $51 $1 $32 $29 $45 $30 $25 $16 $36 $33 $20 $224 $30 $39 $39 $28 $34 $30 $92 $114 $27 $32 $33 $22 $25 $19 $37 $23 $35 $22 $213 $14 $3 $29 $25 $30 $13 $24 $25 $42 $37 $28 $41 $34 $32 $39 $33 $17 $5 $25 $33 $28 $34 $33 $13 $25 $29 $48 $6 $27 $21 $5 $27 $21 $27 $29 $34 $35 $35 $34 $26 $40 $35 $8 $18 $37 $28 $37 $37 $38 $32 $43 $31 $32 $28 $28 $25 $34 $33 $39 $33 $29 $33 $41 $32 $90 $1 $29 $26 

In [15]:
# Feature engineering
# Domain knowledge feature
# The performance of traditional linear regression is not good enough 
# So we add new useful feature
prices_train = np.array([float(item.price) for item in train])
prices_test = np.array([float(item.price) for item in test])
# Apply NLP 
# Add corpus
# Then we come up with the most k-words in corpus and treeat them as seperated features
documents_train = [item.summary for item in train]
documents_test = [item.summary for item in test]

In [16]:
# Encoding
# Apply count vectorizer to form new X_train and X_test
np.random.seed(40)
# Ignore stop English words, We need the important ones
vectorizer = CountVectorizer(max_features=450, stop_words='english')
X_train = vectorizer.fit_transform(documents_train)
X_test = vectorizer.transform(documents_test)

In [17]:
selected_words = vectorizer.get_feature_names_out()
print(f"Number of selected words 'famous' in both train and test: {len(selected_words)}")
print("Selected words in train and test:", selected_words)

Number of selected words 'famous' in both train and test: 450
Selected words in train and test: ['10' '100' '11' '12' '13' '14' '15' '16' '20' '24' '25' '30' '360' '4k'
 '50' 'ac' 'access' 'accessories' 'adapter' 'adhesive' 'adjustable' 'air'
 'airpods' 'aluminum' 'anti' 'apple' 'appliance' 'appliances'
 'approximately' 'art' 'assembly' 'audio' 'auto' 'available' 'backdrop'
 'backed' 'bag' 'bags' 'basket' 'bathroom' 'batteries' 'battery' 'bedding'
 'belt' 'black' 'blue' 'bluetooth' 'board' 'body' 'bpa' 'brand' 'brands'
 'breathable' 'built' 'burner' 'cable' 'cables' 'camera' 'cameras'
 'canvas' 'capacity' 'car' 'carbon' 'card' 'care' 'case' 'category'
 'ceramic' 'certified' 'charger' 'charging' 'christmas' 'clean' 'cleaning'
 'clear' 'closure' 'coffee' 'color' 'colorful' 'colors' 'comes' 'comfort'
 'comfortable' 'compact' 'compatibility' 'compatible' 'components'
 'comprehensive' 'connectivity' 'connector' 'connectors' 'construction'
 'control' 'controls' 'convenient' 'cooling' 'copper

In [18]:
X_train

<16800x450 sparse matrix of type '<class 'numpy.int64'>'
	with 367666 stored elements in Compressed Sparse Row format>

In [19]:
X_test

<2191x450 sparse matrix of type '<class 'numpy.int64'>'
	with 42425 stored elements in Compressed Sparse Row format>

In [20]:
# Modeling
# Traditional linear regression with NLP features
np.random.seed(40)
feature_columns = selected_words
y_train = prices_train
y_test = prices_test

# Train 
model = LinearRegression()
model.fit(X_train, y_train)
for feature, coef in zip(feature_columns, model.coef_):
    print(f"{feature}: {coef: 0.2f}")
print(f"Intercept: {model.intercept_: 0.2f}")

# Predict 
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# Evaluate 
train_mse = mean_squared_error(y_train, y_train_pred)
train_r2 = r2_score(y_train, y_train_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)

print(f"Train MSE: {train_mse: 0.2f}")
print(f"Train R2 Score: {train_r2: 0.2f}%")
print(f"Test MSE: {test_mse: 0.2f}")
print(f"Test R2 Score: {test_r2: 0.2f}%")

10: -1.17
100:  5.42
11:  3.67
12:  1.45
13:  5.55
14:  5.22
15:  16.60
16:  8.29
20:  14.94
24:  33.72
25:  5.14
30:  25.61
360: -8.09
4k:  17.39
50:  10.54
ac: -0.34
access: -3.80
accessories: -8.98
adapter: -6.18
adhesive: -12.70
adjustable:  4.30
air:  1.37
airpods:  0.03
aluminum: -10.82
anti: -1.95
apple:  9.94
appliance:  8.64
appliances:  13.38
approximately: -7.28
art: -11.25
assembly:  16.88
audio:  1.92
auto:  6.13
available: -9.65
backdrop: -19.72
backed: -5.87
bag: -11.76
bags: -10.00
basket: -6.99
bathroom: -4.16
batteries: -3.39
battery: -15.57
bedding:  27.97
belt: -4.56
black: -2.96
blue: -0.49
bluetooth: -6.59
board:  19.00
body: -0.97
bpa: -5.75
brand: -6.54
brands: -12.13
breathable: -0.25
built:  41.30
burner: -4.05
cable: -7.88
cables:  2.37
camera:  2.72
cameras:  11.57
canvas:  6.36
capacity:  15.64
car:  0.83
carbon:  6.26
card: -3.00
care: -6.66
case: -4.20
category: -25.43
ceramic: -4.66
certified:  5.30
charger: -4.26
charging: -5.13
christmas: -2.95
clean: 

In [21]:
# Test dataset
def linear_regression_pricer_nlp(item):
    X_test = vectorizer.transform([item.summary])
    # In case we got - result then it gonna be zero
    return max(model.predict(X_test)[0], 0)

In [22]:
# Test single datapoint
linear_regression_pricer_nlp(items[0])

0

In [23]:
# Evaluation
evaluate(linear_regression_pricer_nlp, test)

  0%|          | 0/2000 [00:00<?, ?it/s]

$42 $77 $14 $13 $59 $19 $6 $24 $9 $3 $26 $23 $48 $28 $12 $1 $38 $4 $332 $81 $7 $77 $50 $38 $8 $28 $2 $32 $11 $5 $20 $6 $8 $728 $3 $71 $43 $4 $65 $20 $90 $177 $36 $37 $30 $34 $21 $16 $3 $6 $25 $55 $41 $14 $9 $7 $32 $49 $66 $8 $36 $25 $22 $54 $136 $111 $20 $3 $26 $33 $4 $15 $34 $28 $18 $23 $25 $33 $29 $188 $51 $1 $31 $112 $29 $11 $6 $37 $29 $14 $10 $11 $10 $24 $23 $13 $2 $51 $45 $1 $25 $13 $12 $42 $7 $67 $33 $55 $26 $15 $29 $31 $55 $99 $3 $76 $9 $53 $12 $63 $43 $25 $40 $27 $29 $15 $11 $11 $12 $50 $13 $44 $159 $77 $25 $6 $2 $12 $17 $14 $45 $34 $7 $8 $6 $18 $6 $38 $6 $125 $27 $3 $12 $40 $34 $31 $23 $19 $30 $16 $45 $49 $19 $9 $84 $246 $35 $87 $59 $30 $20 $105 $105 $85 $6 $158 $15 $16 $55 $8 $13 $19 $11 $11 $114 $19 $9 $63 $11 $8 $65 $9 $85 $9 $12 $91 $60 $8 $19 $37 $23 $5 $40 $62 $15 $47 $64 $25 $20 $41 $9 $6 $11 $14 $21 $105 $37 $36 $94 $6 $7 $33 $15 $43 $1 $32 $45 $25 $38 $3 $24 $32 $8 $2 $48 $34 $116 $7 $8 $55 $25 $1 $24 $3 $29 $21 $20 $26 $119 $340 $20 $37 $23 $8 $17 $508 $116 $20 $40 $

In [24]:
# Modeling
# Random forest with NLP features
np.random.seed(40)
y_train = prices_train
y_test = prices_test

# Train 
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=4)
model.fit(X_train, y_train)

# Predict 
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# Evaluate 
train_mse = mean_squared_error(y_train, y_train_pred)
train_r2 = r2_score(y_train, y_train_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)

print(f"Train MSE: {train_mse: 0.2f}")
print(f"Train R2 Score: {train_r2: 0.2f}%")
print(f"Test MSE: {test_mse: 0.2f}")
print(f"Test R2 Score: {test_r2: 0.2f}%")

Train MSE:  806.38
Train R2 Score:  0.93%
Test MSE:  4098.00
Test R2 Score:  0.34%


In [25]:
# Test dataset
def random_forest(item):
    X_test = vectorizer.transform([item.summary])
    # In case we got - result then it gonna be zero
    return max(model.predict(X_test)[0], 0)  

In [26]:
# Test single datapoint
random_forest(items[0])

9.209200000000008

In [27]:
# Evaluation
evaluate(random_forest, test)

  0%|          | 0/2000 [00:00<?, ?it/s]

$7 $55 $10 $5 $6 $6 $0 $11 $2 $9 $6 $29 $13 $6 $26 $2 $8 $6 $297 $76 $8 $6 $57 $8 $8 $11 $3 $8 $0 $9 $4 $24 $27 $740 $2 $16 $19 $6 $24 $3 $12 $110 $10 $3 $22 $8 $9 $21 $1 $1 $13 $15 $21 $0 $6 $5 $27 $7 $10 $8 $21 $8 $17 $15 $148 $154 $16 $3 $13 $2 $9 $2 $7 $35 $13 $18 $56 $8 $36 $212 $15 $2 $6 $111 $14 $89 $4 $19 $11 $17 $10 $1 $12 $9 $14 $11 $0 $37 $21 $2 $17 $94 $21 $13 $1 $27 $26 $3 $11 $3 $128 $18 $44 $90 $15 $52 $16 $100 $16 $10 $20 $11 $10 $4 $88 $29 $20 $7 $7 $141 $8 $18 $203 $43 $4 $3 $1 $17 $58 $1 $5 $16 $14 $4 $6 $6 $7 $13 $10 $15 $9 $29 $3 $1 $16 $12 $26 $1 $70 $3 $3 $30 $4 $2 $62 $250 $9 $13 $32 $7 $6 $9 $43 $119 $8 $141 $44 $25 $2 $2 $13 $3 $10 $5 $144 $10 $29 $18 $4 $7 $9 $1 $34 $7 $12 $7 $27 $0 $13 $19 $25 $4 $18 $9 $10 $6 $62 $35 $12 $3 $10 $24 $78 $5 $2 $156 $18 $1 $21 $1 $35 $1 $4 $48 $2 $45 $19 $9 $13 $5 $17 $5 $18 $8 $28 $20 $9 $33 $8 $7 $12 $4 $0 $9 $10 $10 $8 $17 $59 $292 $39 $39 $14 $1 $43 $463 $86 $14 $0 $19 $1 $11 $16 $7 $12 $12 $7 $27 $30 $37 $24 $52 $13 $1 $1

In [28]:
# Modeling
# XGBoost with NLP features
np.random.seed(40)
feature_columns = selected_words
y_train = prices_train
y_test = prices_test

# Train 
model = xgb.XGBRegressor(n_estimators=1000, random_state=42, n_jobs=4, learning_rate=0.1)
model.fit(X_train, y_train)

# Predict 
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# Evaluate 
train_mse = mean_squared_error(y_train, y_train_pred)
train_r2 = r2_score(y_train, y_train_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)

print(f"Train MSE: {train_mse: 0.2f}")
print(f"Train R2 Score: {train_r2: 0.2f}%")
print(f"Test MSE: {test_mse: 0.2f}")
print(f"Test R2 Score: {test_r2: 0.2f}%")

Train MSE:  742.03
Train R2 Score:  0.93%
Test MSE:  4209.67
Test R2 Score:  0.32%


In [29]:
# Test dataset
def xgb(item):
    X_test = vectorizer.transform([item.summary])
    # In case we got - result then it gonna be zero
    return max(model.predict(X_test)[0], 0)  

In [30]:
# Test single datapoint
xgb(items[0])

9.031963

In [31]:
# Evaluation
evaluate(xgb, test)

  0%|          | 0/2000 [00:00<?, ?it/s]

$35 $103 $15 $5 $4 $23 $2 $17 $1 $4 $15 $14 $30 $11 $13 $1 $0 $3 $307 $113 $49 $26 $19 $19 $3 $9 $7 $21 $6 $0 $3 $13 $8 $708 $7 $49 $29 $13 $4 $13 $110 $134 $11 $6 $67 $23 $26 $4 $1 $1 $30 $15 $27 $1 $0 $20 $11 $6 $17 $14 $29 $21 $22 $49 $64 $114 $12 $8 $5 $13 $11 $0 $11 $8 $11 $24 $85 $21 $36 $211 $14 $0 $14 $135 $31 $79 $3 $32 $2 $9 $14 $3 $2 $8 $14 $3 $6 $22 $24 $3 $29 $57 $11 $15 $35 $35 $13 $37 $15 $1 $216 $10 $60 $57 $12 $18 $16 $83 $4 $0 $42 $8 $77 $2 $87 $24 $81 $9 $10 $184 $12 $3 $134 $22 $16 $13 $1 $132 $69 $3 $13 $24 $2 $15 $8 $2 $5 $29 $14 $57 $8 $17 $22 $11 $44 $20 $16 $18 $65 $14 $10 $43 $3 $16 $67 $247 $42 $41 $30 $11 $6 $27 $4 $158 $13 $87 $8 $0 $28 $1 $11 $10 $13 $17 $135 $40 $45 $36 $5 $4 $62 $10 $56 $4 $22 $36 $7 $4 $10 $39 $8 $6 $11 $58 $9 $22 $30 $13 $18 $1 $14 $48 $110 $7 $12 $127 $22 $9 $25 $7 $29 $28 $22 $37 $1 $43 $24 $30 $7 $8 $34 $18 $10 $12 $3 $17 $123 $35 $5 $6 $15 $5 $1 $3 $26 $21 $13 $7 $61 $261 $64 $31 $27 $6 $3 $507 $73 $9 $14 $21 $5 $1 $28 $9 $11 $14 $